# 누리철도 HR 챗봇 — 규정 검색(RAG) 구축

이 노트북 하나로 끝까지 돌아가요 (다른 코드 업로드 필요 없음).

1. 저장소 받기: [DB](https://github.com/Jaeyong-s-HR-Chatbot/DB)(규정·법령 조각 파일), [goldenset](https://github.com/Jaeyong-s-HR-Chatbot/goldenset)(데이터 형식·라우터)
2. `rag` 패키지 파일 만들기 (`%%writefile` 셀들 — 코드 고칠 땐 여기서 고치고 다시 실행)
3. 파이프라인 구축: BGE-M3 임베딩 + BM25 색인
4. 질문 하나가 ① 쪼개기 → ② 분류 → ③ 라우터 → ④ 검색 → `LlmRequest` 까지 가는 과정 보기

**Colab**: 런타임 → 런타임 유형 변경 → GPU(L4/T4). DB(docker)는 필요 없어요.
처음 한 번은 BGE-M3 다운로드(약 2.3GB) + 조각 약 2,600개 임베딩으로 몇 분 걸리고, 임베딩은 `RAG_CACHE_DIR` 에 저장돼 다음부터 바로 불러와요.

## 0. 준비

In [ ]:
import os, subprocess, sys
from pathlib import Path

BASE = Path("/content") if Path("/content").exists() else Path.cwd()
ORG = "https://github.com/Jaeyong-s-HR-Chatbot"
for name in ["DB", "goldenset"]:
    d = BASE / name
    if d.exists():
        subprocess.run(["git", "-C", str(d), "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", f"{ORG}/{name}", str(d)], check=True)

WORK = BASE / "rag_build"                     # rag 패키지가 만들어질 폴더
(WORK / "rag").mkdir(parents=True, exist_ok=True)
os.environ["DB_DIR"] = str(BASE / "DB")
os.environ["GOLDENSET_DIR"] = str(BASE / "goldenset")
os.environ.setdefault("RAG_CACHE_DIR", str(BASE / "rag_cache"))   # 드라이브에 두면 런타임이 바뀌어도 재사용: "/content/drive/MyDrive/rag_cache"
# os.environ["RAG_EMBEDDER"] = "mock"   # GPU 없이 코드 흐름만 볼 때 (점수는 의미 없음)

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("작업 폴더:", Path.cwd())

In [ ]:
!pip install -q "sentence-transformers>=3" "pydantic>=2" pandas numpy
!pip install -q kiwipiepy || echo "kiwipiepy 설치 실패 → BM25는 어절+2-gram 으로 대신해요"

## 1. rag 패키지

| 파일 | 하는 일 |
|---|---|
| `config.py` | 설정, 검색 구성(A0 Dense만 / A1 +BM25 / B +하드 필터 / C +confidence 따라 하드·소프트, 기본) |
| `goldenset.py` | goldenset 저장소의 `ChunkHit`·`LlmRequest`·`build_messages`·라우터 불러오기 |
| `split.py` | ① 복합 질문 쪼개기 (규칙) |
| `classify.py` | ② 카테고리 분류 (BGE-M3 프로토타입 + 키워드), slots, 범위 밖·애매함·민감 |
| `embed.py` | BGE-M3 임베딩 (mock 대체 가능) |
| `retrieve.py` | ④ Dense + BM25 → RRF, 카테고리 필터, 내부규정 우선, 후보 20개 |
| `pipeline.py` | ③ 라우터 + 전체 묶기, 상위 8개 → `LlmRequest` |

In [ ]:
%%writefile rag/__init__.py
"""누리철도 HR 챗봇 규정 검색(RAG) 베이스라인.

    from rag import Pipeline, load_config, split_question
    pipe = Pipeline(load_config())
    r = pipe.run("결혼하면 휴가 며칠이고 경조금 얼마야?")
"""

from .config import PRESETS, Config, load_config
from .pipeline import Pipeline, PipelineResult, SubResult
from .split import split_question

__all__ = ["PRESETS", "Config", "load_config", "Pipeline", "PipelineResult", "SubResult", "split_question"]

In [ ]:
%%writefile rag/config.py
"""설정. 모델·색인은 그대로 두고 스위치(검색 방식·필터)만 바꿔 실험한다.

경로는 환경변수로 바꿀 수 있다.
- DB_DIR         : DB 저장소 (data/regs/chunks.jsonl, data/laws/chunks.jsonl 이 있는 곳)
- GOLDENSET_DIR  : goldenset 저장소 (contracts/, router/)
- RAG_CACHE_DIR  : 임베딩 캐시 저장 위치
- RAG_EMBEDDER   : "bge-m3"(기본) / "mock"(모델 없이 코드 흐름만 점검)
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field, replace
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent   # 이 저장소(RAG) 루트


def _first_dir(*cands: str | Path | None) -> Path:
    for c in cands:
        if c and Path(c).is_dir():
            return Path(c).resolve()
    return Path(next(c for c in cands if c))   # 없어도 일단 첫 후보 (나중에 친절한 오류)


# 실험 구성. 위에서부터 하나씩 더해진다.
PRESETS: dict[str, dict] = {
    "A0": dict(use_dense=True, use_bm25=False, filter_mode="none"),       # Dense만
    "A1": dict(use_dense=True, use_bm25=True, filter_mode="none"),        # + BM25, RRF로 합치기
    "B":  dict(use_dense=True, use_bm25=True, filter_mode="hard"),        # + 분류 결과로 하드 필터
    "C":  dict(use_dense=True, use_bm25=True, filter_mode="adaptive"),    # + confidence 따라 하드/소프트, 약하면 해제 (최종)
}


@dataclass(frozen=True)
class Config:
    # 경로
    db_dir: Path
    goldenset_dir: Path
    cache_dir: Path
    data_subdirs: tuple[str, ...] = ("data/regs", "data/laws")

    # 모델
    embedder: str = "bge-m3"          # "bge-m3" / "mock"
    model_name: str = "BAAI/bge-m3"
    max_seq_len: int = 1024           # 메모리 부족하면 512
    batch_size: int = 16

    # 검색 스위치 (preset 으로 바꿈)
    preset_name: str = "C"
    use_dense: bool = True
    use_bm25: bool = True
    filter_mode: str = "adaptive"     # none / hard / adaptive

    # 검색 숫자
    n_candidates: int = 50            # Dense·BM25 각각 뽑는 후보 수
    rrf_k: int = 60
    pool: int = 20                    # 화면에 보여줄 후보 수 (검색 후보 최대 20개)
    top_k: int = 8                    # LLM에 넘길 조각 수 (LlmRequest 최대 8)
    min_filtered: int = 3             # 하드 필터 뒤 후보가 이보다 적으면 필터 해제
    soft_boost: float = 0.3           # 소프트 필터: 카테고리 맞는 조각 RRF 점수 가산 비율
    reg_boost: float = 0.1            # 내부규정 우선 (법령이 규정을 밀어내지 않게)
    exclude_docs: tuple[str, ...] = ("r18",)   # 임원 복무 지침: 직원 답변에 섞이면 안 됨

    # 분류 임계값 (bge-m3 코사인 기준)
    proto_margin: float = 0.03        # 1등 카테고리와 이만큼 가까우면 같이 붙임 (멀티라벨)
    high_conf: float = 0.50           # 이 이상 + 1·2등 차이가 conf_gap 이상이면 confident
    conf_gap: float = 0.04
    in_scope: float = 0.35            # 1등 카테고리 점수가 이보다 낮으면 범위 밖 후보
    fallback_cos: float = 0.40        # 필터 검색 1등 코사인이 이보다 낮으면 필터 풀고 재검색

    extra: dict = field(default_factory=dict)

    def preset(self, name: str) -> "Config":
        if name not in PRESETS:
            raise KeyError(f"모르는 구성 {name!r}. 가능한 값: {list(PRESETS)}")
        return replace(self, preset_name=name, **PRESETS[name])

    def replace(self, **kw) -> "Config":
        return replace(self, **kw)

    @property
    def chunk_files(self) -> list[Path]:
        return [self.db_dir / d / "chunks.jsonl" for d in self.data_subdirs]


def load_config(preset: str = "C", **overrides) -> Config:
    env = os.environ.get
    db_dir = _first_dir(env("DB_DIR"), ROOT.parent / "DB", "/content/DB", ROOT / "DB")
    gs_dir = _first_dir(env("GOLDENSET_DIR"), ROOT.parent / "goldenset", "/content/goldenset", ROOT / "goldenset")
    cache = Path(env("RAG_CACHE_DIR") or (ROOT / "cache"))
    cfg = Config(db_dir=db_dir, goldenset_dir=gs_dir, cache_dir=cache,
                 embedder=env("RAG_EMBEDDER", "bge-m3"))
    cfg = cfg.preset(preset)
    return replace(cfg, **overrides) if overrides else cfg

In [ ]:
%%writefile rag/goldenset.py
"""goldenset 저장소(contracts/, router/)를 불러온다.

데이터 형식(ChunkHit, LlmRequest)과 build_messages, 라우터는 goldenset 저장소에 있는 걸 그대로 쓴다.
위치는 환경변수 GOLDENSET_DIR (없으면 ../goldenset, /content/goldenset 순서로 찾음).
"""

from __future__ import annotations

import sys

from .config import load_config

_dir = load_config().goldenset_dir
if not (_dir / "contracts" / "schemas.py").is_file():
    raise ImportError(
        f"goldenset 저장소를 못 찾았어요: {_dir}\n"
        "  git clone https://github.com/Jaeyong-s-HR-Chatbot/goldenset 한 뒤 "
        "os.environ['GOLDENSET_DIR'] = '<그 경로>' 로 알려 주세요."
    )
if str(_dir) not in sys.path:
    sys.path.insert(0, str(_dir))

from contracts.pipeline import (  # noqa: E402
    SYSTEM_PROMPT, build_messages, chunk_hit_from_row, parse_llm_answer, to_chat_response,
)
from contracts.schemas import ChunkHit, LlmRequest  # noqa: E402
from router import Route, route  # noqa: E402

__all__ = [
    "SYSTEM_PROMPT", "build_messages", "chunk_hit_from_row", "parse_llm_answer", "to_chat_response",
    "ChunkHit", "LlmRequest", "Route", "route",
]

In [ ]:
%%writefile rag/split.py
"""① 질문 쪼개기 (규칙, LLM 안 씀).

"결혼하면 휴가 며칠이고 경조금 얼마야?" → ["결혼하면 휴가 며칠인가요?", "결혼하면 경조금 얼마야?"]
"복지포인트는 언제 들어오고 어디서 쓸 수 있어요?" → ["복지포인트는 언제 들어오나요?", "복지포인트는 어디서 쓸 수 있어요?"]
"5일 썼는데, 며칠까지 돼요?" → 그대로 (앞 절은 상황 설명이지 따로 묻는 게 아님)

규칙
- '…고 ' / '…이고 ' / '그리고' 로 이어진 두 절이 **둘 다 의문사(며칠·얼마·언제·어디·어떻게 …)** 를 가질 때만 나눈다.
- 앞 절 맨 앞의 조건("결혼하면", "출산 후에")·주제("복지포인트는")는 뒤 절에 없으면 이어 붙인다.
"""

from __future__ import annotations

import re

WH = re.compile(r"(며칠|몇|얼마|언제|어디|어떻게|어떤|무엇|뭐|뭘|누가|누구|왜|있어|있나|되나|돼|가능|나와|받아|받나)")
# 연결: "며칠이고 ", "들어오고 ", "그리고 ", "…하고 "
_JOIN = re.compile(r"(?<=[가-힣])(이고|고)\s+|\s*그리고\s+|\s*,\s*그리고\s+")
_COND = re.compile(r"^(\S+(?:하면|되면|이면|으면|면|때|후에|전에|중에|경우))\s+")
_TOPIC = re.compile(r"^(\S+(?:은|는))\s+")


def _close(head: str, joiner: str) -> str:
    """'며칠이' + '고' → '며칠인가요?', '들어오' + '고' → '들어오나요?'"""
    head = head.rstrip(" ,")
    if joiner == "이고":
        return head + "인가요?"
    if joiner == "고":
        return head + "나요?"
    return head if head.endswith("?") else head + "?"


def _carry(first: str, second: str) -> str:
    """앞 절의 조건·주제를 뒤 절에 이어 붙인다 (뒤 절에 이미 있으면 안 붙임)."""
    prefix = []
    rest = first
    for pat in (_COND, _TOPIC):
        m = pat.match(rest)
        if m:
            prefix.append(m.group(1))
            rest = rest[m.end():]
    if not prefix:
        return second
    if _COND.match(second) or _TOPIC.match(second):
        return second
    return " ".join(prefix) + " " + second


def split_question(q: str) -> list[str]:
    q = re.sub(r"\s+", " ", q.strip())
    m = _JOIN.search(q)
    if not m:
        return [q]
    head, tail = q[: m.start()], q[m.end():]
    joiner = (m.group(1) or "그리고").strip()
    if not (WH.search(head) and WH.search(tail)) or len(tail) < 3:
        return [q]
    first = _close(head, joiner)
    second = _carry(first, tail)
    return [first] + split_question(second)[: 3]   # 최대 4개

In [ ]:
%%writefile rag/classify.py
"""② 질문 분류 (BGE-M3 프로토타입 + 규칙, LLM 안 씀).

- 업무 영역(멀티라벨): 카테고리마다 설명·예시 질문을 임베딩해 평균(프로토타입)을 만들고 코사인으로 비교.
  키워드가 맞으면 점수를 더한다. 1등과 proto_margin 안쪽이면 같이 붙인다.
- slots: 이벤트·대상·지역·숫자 같은 조건 (정규식)
- 범위 밖 / 애매함(되묻기) / 민감(담당자 연결) 표시
카테고리 이름은 DB 저장소 rules/regulations.yaml 과 같다.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np

# 카테고리 → (키워드, 예시 질문). 키워드는 DB rules/regulations.yaml category_rules 와 같게 유지.
CATEGORIES: dict[str, tuple[list[str], list[str]]] = {
    "근태관리": (
        ["휴가", "연차", "반차", "근무시간", "휴일", "휴게", "병가", "지각", "조퇴", "당직", "유연근무", "시차출퇴근", "교대근무", "출근", "퇴근"],
        ["연차휴가는 며칠 나와요?", "병가는 1년에 며칠까지 쓸 수 있어?", "반차 쓰면 연차에서 얼마나 빠져요?",
         "출근 시간을 늦추는 유연근무 신청할 수 있어?", "지각하면 어떻게 처리돼요?"],
    ),
    "수당": (
        ["수당", "보수", "봉급", "급여", "월급", "상여", "임금", "성과연봉", "통상임금", "야근", "시간외", "야간", "가족수당"],
        ["시간외근무수당은 어떻게 계산해?", "월급날이 언제야?", "가족수당은 얼마 나와요?",
         "야간근무하면 수당이 얼마예요?", "성과연봉은 언제 지급돼요?"],
    ),
    "교육": (
        ["교육", "훈련", "연수", "학점", "수료", "강사", "법정교육", "예방교육", "대학원"],
        ["교육 수료 기준 점수가 몇 점이야?", "법정 의무교육은 꼭 들어야 해?", "사내 강사로 나가면 강사료 나와요?",
         "국외훈련 지원 자격이 어떻게 돼?", "안전교육은 몇 시간 받아야 해?"],
    ),
    "출장": (
        ["출장", "여비", "일비", "숙박비", "식비", "교통비", "국외여비", "운임"],
        ["서울 출장 가면 숙박비 얼마까지 나와?", "출장 일비는 직급별로 얼마예요?", "국외출장 여비 기준이 뭐야?",
         "1박 2일 출장 가면 얼마 받아요?", "출장 교통비는 실비로 줘요?"],
    ),
    "기타 복리후생": (
        ["복지", "포인트", "복지포인트", "학자금", "대부", "생활안정", "주택", "의료비", "건강검진", "동호회", "휴양", "콘도"],
        ["복지포인트는 몇 점 주나요?", "자녀 학자금 지원 받을 수 있어?", "생활안정자금 대부 한도가 얼마야?",
         "건강검진은 회사에서 해줘요?", "휴양소 이용 신청은 어떻게 해?"],
    ),
    "생애이벤트-결혼": (
        ["결혼", "혼인", "청원휴가", "경조", "경조금", "경조사", "사망", "조사", "장례", "부고"],
        ["결혼하면 휴가 며칠 받아요?", "결혼 경조금 얼마 나와?", "부모님 돌아가시면 경조휴가 며칠이야?",
         "자녀 결혼하면 휴가 있어요?", "경조사 휴가는 언제까지 써야 해?"],
    ),
    "생애이벤트-출산": (
        ["출산", "육아", "임신", "난임", "태아", "유산", "사산", "입양", "자녀돌봄", "배우자출산", "모성"],
        ["배우자가 출산하면 휴가 있어?", "출산전후휴가는 며칠이야?", "임신 중에 근무시간 줄일 수 있어?",
         "난임치료 휴가 있어요?", "육아기 근로시간 단축 신청할 수 있어?"],
    ),
    "생애이벤트-휴직·복직·퇴직": (
        ["휴직", "복직", "퇴직", "퇴직금", "정년", "명예퇴직", "임금피크", "임피", "육아휴직", "사직"],
        ["육아휴직은 몇 년까지 가능해?", "퇴직금은 어떻게 계산해?", "정년이 몇 살이에요?",
         "임금피크제 들어가면 월급이 얼마나 줄어?", "휴직 끝나고 복직 신청은 언제 해?"],
    ),
    "생애이벤트-입사·승진": (
        ["승진", "승급", "근무성적", "평정", "승격", "채용", "신규임용", "수습", "임용", "입사", "시험승진"],
        ["승진 평정 점수 비율이 어떻게 돼?", "수습 기간은 얼마나 돼?", "4급 시험승진 응시 자격이 뭐야?",
         "승진 최소 소요연수가 몇 년이야?", "신규 채용 절차가 어떻게 돼?"],
    ),
}
COMMON = "공통"

# 범위 밖: HR·규정과 무관한 주제
OUT_OF_SCOPE = re.compile(
    r"(주식|코인|비트코인|부동산 투자|로또|날씨|맛집|점심 메뉴|연애|게임|축구|야구|노래 추천|영화 추천|여행지 추천|주가|환율|선거|정치)"
)
# 민감: 담당자 연결이 맞는 주제 (검색은 하되 표시)
SENSITIVE = re.compile(r"(성희롱|성폭력|괴롭힘|갑질|징계|신고|고충|폭행|자살|우울)")

# 애매함: 이 일반 명사만 있고 무엇인지 안 정해지면 되묻기
AMBIGUOUS: dict[str, list[str]] = {
    "휴가": ["연차휴가", "경조휴가(결혼·사망)", "병가", "출산전후휴가·배우자출산휴가"],
    "수당": ["시간외·야간·휴일근무수당", "가족수당", "연차휴가 미사용수당", "당직비"],
    "휴직": ["육아휴직", "질병휴직", "가족돌봄휴직"],
    "교육": ["법정 의무교육", "직무교육·집합교육", "국외훈련"],
}
_QUALIFIER = re.compile(
    r"(연차|경조|결혼|출산|병가|배우자|육아|청원|포상|공가|특별|가족|시간외|야간|휴일|연장|미사용|당직|질병|돌봄|"
    r"법정|안전|직무|집합|국외|성희롱|예방|사망|임신|난임|입양|여름|하계|장기)"
)

# slots
_EVENT = re.compile(r"(결혼|혼인|출산|사망|돌아가|장례|입양|임신|유산|입사|승진|퇴직|정년|휴직|복직|이사|전보)")
_RELATION = re.compile(r"(본인|배우자|아내|남편|와이프|자녀|아들|딸|부모|아버지|어머니|엄마|아빠|조부모|할머니|할아버지|형제|자매|장인|장모|시부모)")
_REGION = re.compile(r"(서울|부산|대구|인천|광주|대전|울산|세종|수도권|광역시|충청|영남|호남|강원|제주|국외|해외)")
_NUMBER = re.compile(r"(\d+(?:\.\d+)?)\s*(일|박|시간|년|개월|만원|원|급|세|살|회|명)")
_GRADE = re.compile(r"([1-6])\s*급")


@dataclass
class QueryClass:
    categories: list[tuple[str, float]]          # 붙인 카테고리 (멀티라벨, 점수 순)
    scores: dict[str, float]                     # 전체 카테고리 점수
    confident: bool                              # 하드 필터를 걸어도 될 만큼 확실한가
    slots: dict[str, list[str]] = field(default_factory=dict)
    in_scope: bool = True
    ambiguous: bool = False
    clarify_options: list[str] = field(default_factory=list)
    sensitive: bool = False
    keyword_hits: dict[str, list[str]] = field(default_factory=dict)

    @property
    def category_names(self) -> list[str]:
        return [c for c, _ in self.categories]


def extract_slots(q: str) -> dict[str, list[str]]:
    slots = {
        "event": _EVENT.findall(q),
        "relation": _RELATION.findall(q),
        "region": _REGION.findall(q),
        "number": ["".join(m) for m in _NUMBER.findall(q)],
        "grade": [g + "급" for g in _GRADE.findall(q)],
    }
    return {k: list(dict.fromkeys(v)) for k, v in slots.items() if v}


class Classifier:
    def __init__(self, embedder, cfg):
        self.cfg = cfg
        self.embedder = embedder
        self.names = list(CATEGORIES)
        protos = []
        for name, (kws, examples) in CATEGORIES.items():
            texts = [f"{name}: {', '.join(kws)}"] + examples
            v = embedder.encode(texts, is_query=True).mean(axis=0)
            protos.append(v / (np.linalg.norm(v) + 1e-9))
        self.protos = np.stack(protos)

    def with_config(self, cfg) -> "Classifier":
        new = object.__new__(Classifier)
        new.__dict__.update(self.__dict__, cfg=cfg)
        return new

    def __call__(self, q: str, qvec: np.ndarray | None = None) -> QueryClass:
        cfg = self.cfg
        if qvec is None:
            qvec = self.embedder.encode([q], is_query=True)[0]
        cos = self.protos @ qvec
        hits = {n: [k for k in CATEGORIES[n][0] if k in q] for n in self.names}
        hits = {n: v for n, v in hits.items() if v}
        scores = {n: float(c) + (0.08 if n in hits else 0.0) for n, c in zip(self.names, cos)}
        ranked = sorted(scores.items(), key=lambda kv: -kv[1])
        top, top_s = ranked[0]
        second_s = ranked[1][1]
        cats = [(n, s) for n, s in ranked if s >= top_s - cfg.proto_margin]
        cats += [(n, scores[n]) for n in hits if n not in dict(cats)]   # 키워드가 맞은 카테고리는 항상 붙임

        in_scope = not OUT_OF_SCOPE.search(q) and (top_s >= cfg.in_scope or bool(hits))
        confident = in_scope and (
            (top_s >= cfg.high_conf and top_s - second_s >= cfg.conf_gap) or (len(hits) == 1 and top in hits)
        )

        # 애매함: '휴가 며칠이야?'처럼 짧고, 일반 명사만 있고 종류를 가르는 말이 없음
        ambiguous, options = False, []
        short = len(re.sub(r"[\s?!.,]", "", q)) <= 10
        for word, opts in AMBIGUOUS.items():
            if short and word in q and not _QUALIFIER.search(q) and not extract_slots(q).get("event"):
                ambiguous, options = True, opts
                break
        if ambiguous:
            confident = False

        return QueryClass(
            categories=cats, scores=scores, confident=confident, slots=extract_slots(q),
            in_scope=in_scope, ambiguous=ambiguous and in_scope, clarify_options=options,
            sensitive=bool(SENSITIVE.search(q)), keyword_hits=hits,
        )

In [ ]:
%%writefile rag/embed.py
"""임베딩 모델. BGE-M3 (dense 1024차원, 정규화 → 내적 = 코사인).

RAG_EMBEDDER=mock 이면 모델 다운로드 없이 글자 n-gram 해시 벡터를 쓴다 (코드 흐름 점검용, 점수는 의미 없음).
"""

from __future__ import annotations

import hashlib
import re

import numpy as np


class BgeM3Embedder:
    name = "bge-m3"

    def __init__(self, cfg):
        import torch
        from sentence_transformers import SentenceTransformer

        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(cfg.model_name, device=device)
        if device == "cuda":
            self.model.half()
        self.model.max_seq_length = cfg.max_seq_len
        self.batch_size = cfg.batch_size
        self.device = device
        self.dim = self.model.get_sentence_embedding_dimension()

    def encode(self, texts: list[str], is_query: bool = False, show_progress: bool = False) -> np.ndarray:
        v = self.model.encode(
            list(texts), batch_size=self.batch_size, normalize_embeddings=True,
            show_progress_bar=show_progress, convert_to_numpy=True,
        )
        return v.astype(np.float32)


class MockEmbedder:
    """글자 2·3-gram 해시 → 고정 차원. 같은 글자가 많이 겹치면 코사인이 높다."""
    name = "mock"

    def __init__(self, cfg=None, dim: int = 1024):
        self.dim = dim
        self.device = "cpu"

    def _vec(self, t: str) -> np.ndarray:
        v = np.zeros(self.dim, dtype=np.float32)
        s = re.sub(r"\s+", " ", t)
        for n in (2, 3):
            for i in range(len(s) - n + 1):
                g = s[i:i + n]
                if " " in g:
                    continue
                h = int(hashlib.md5(g.encode()).hexdigest()[:8], 16)
                v[h % self.dim] += 1.0
        return v / (np.linalg.norm(v) + 1e-9)

    def encode(self, texts: list[str], is_query: bool = False, show_progress: bool = False) -> np.ndarray:
        return np.stack([self._vec(t) for t in texts]) if texts else np.zeros((0, self.dim), np.float32)


def make_embedder(cfg):
    if cfg.embedder == "mock":
        return MockEmbedder(cfg)
    return BgeM3Embedder(cfg)

In [ ]:
%%writefile rag/retrieve.py
"""④ 검색기: (필터) → Dense + BM25 → RRF → 후보 최대 20개.

검색 대상은 DB 저장소의 조각 파일(data/regs, data/laws 의 chunks.jsonl) 중 searchable 인 것.
DB(docker)가 없어도 돌아가게 파일에서 바로 읽고, 임베딩은 cache_dir 에 저장해 두 번째부터는 바로 불러온다.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
from collections import Counter
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np

from .classify import COMMON, QueryClass


# ---------------------------------------------------------------- 조각 읽기
def load_rows(cfg) -> dict[str, dict]:
    rows: dict[str, dict] = {}
    missing = [p for p in cfg.chunk_files if not p.is_file()]
    if len(missing) == len(cfg.chunk_files):
        raise FileNotFoundError(
            f"조각 파일이 없어요: {[str(p) for p in missing]}\n"
            "  git clone https://github.com/Jaeyong-s-HR-Chatbot/DB 한 뒤 "
            "os.environ['DB_DIR'] = '<그 경로>' 로 알려 주세요."
        )
    for p in cfg.chunk_files:
        if not p.is_file():
            continue
        with open(p, encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                if not r.get("searchable", True) or r["document_id"] in cfg.exclude_docs:
                    continue
                rows[r["chunk_id"]] = r
    return rows


def embed_text(r: dict) -> str:
    """DB rag_embed.py 와 같은 입력: '규정명 조문 제목\\n본문'"""
    head = " ".join(x for x in (r["doc_name"], r.get("article") or "", r.get("article_title") or "") if x)
    return f"{head}\n{r['text']}"


def title_of(r: dict) -> str:
    head = f"{r['doc_name']} {r.get('article') or ''}".strip()
    return f"{head}({r['article_title']})" if r.get("article") and r.get("article_title") else head


# ---------------------------------------------------------------- BM25
_WORD = re.compile(r"[0-9A-Za-z가-힣]+")
_JOSA = re.compile(r"(은|는|이|가|을|를|의|에|에서|으로|로|와|과|도|만|까지|부터|이나|나|요|이야|야|예요|이에요)$")


def _kiwi():
    try:
        from kiwipiepy import Kiwi
        return Kiwi()
    except Exception:
        return None


class BM25:
    """한국어 BM25. kiwipiepy 가 있으면 형태소(명사·동사 어간), 없으면 어절 + 글자 2-gram."""

    def __init__(self, docs: list[str], k1: float = 1.2, b: float = 0.75):
        self.kiwi = _kiwi()
        self.k1, self.b = k1, b
        toks = [self.tokenize(d) for d in docs]
        self.tf = [Counter(t) for t in toks]
        self.len = np.array([len(t) for t in toks], dtype=np.float32)
        self.avg = float(self.len.mean()) if len(toks) else 1.0
        df = Counter(w for t in self.tf for w in t)
        n = len(toks)
        self.idf = {w: math.log(1 + (n - c + 0.5) / (c + 0.5)) for w, c in df.items()}
        self.index: dict[str, list[int]] = {}
        for i, t in enumerate(self.tf):
            for w in t:
                self.index.setdefault(w, []).append(i)

    def tokenize(self, s: str) -> list[str]:
        if self.kiwi is not None:
            return [t.form for t in self.kiwi.tokenize(s) if t.tag[:2] in ("NN", "VV", "VA", "XR", "SL", "SN") or t.tag == "NR"]
        out = []
        for w in _WORD.findall(s):
            w = _JOSA.sub("", w) or w
            out.append(w)
            out += [w[i:i + 2] for i in range(len(w) - 1)] if len(w) > 2 else []
        return out

    def scores(self, q: str) -> np.ndarray:
        s = np.zeros(len(self.tf), dtype=np.float32)
        for w in set(self.tokenize(q)):
            idf = self.idf.get(w)
            if idf is None:
                continue
            for i in self.index[w]:
                f = self.tf[i][w]
                s[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return s


# ---------------------------------------------------------------- 결과 형식
@dataclass
class Candidate:
    chunk_id: str
    score: float            # 최종 점수 (RRF + 가산)
    cos: float              # 질문과의 코사인 (Dense)
    dense_rank: int | None
    bm25_rank: int | None
    doc_kind: str
    categories: list[str]
    in_filter: bool

    def to_dict(self) -> dict:
        d = asdict(self)
        d["score"] = round(d["score"], 5)
        d["cos"] = round(d["cos"], 4)
        return d


@dataclass
class SearchResult:
    candidates: list[Candidate]
    used_filter: bool
    filter_categories: list[str]
    filter_kind: str                     # none / hard / soft
    used_fallback: bool
    top_cos: float
    doc_kind_counts: dict[str, int] = field(default_factory=dict)


# ---------------------------------------------------------------- 검색기
class Retriever:
    def __init__(self, cfg, embedder, rows: dict[str, dict] | None = None, matrix: np.ndarray | None = None,
                 bm25: BM25 | None = None):
        self.cfg = cfg
        self.embedder = embedder
        self.rows = rows if rows is not None else load_rows(cfg)
        self.ids = list(self.rows)
        self.pos = {c: i for i, c in enumerate(self.ids)}
        self.cats = [set(self.rows[c].get("categories") or [COMMON]) for c in self.ids]
        self.is_reg = np.array([self.rows[c].get("doc_kind") != "법령" for c in self.ids])
        self.matrix = matrix if matrix is not None else self._load_or_embed()
        self.bm25 = bm25 if bm25 is not None else BM25([embed_text(self.rows[c]) for c in self.ids])

    def with_config(self, cfg) -> "Retriever":
        """모델·색인은 그대로, 스위치만 바꾼 새 검색기."""
        return Retriever(cfg, self.embedder, self.rows, self.matrix, self.bm25)

    def _load_or_embed(self) -> np.ndarray:
        cfg = self.cfg
        key = hashlib.md5(("|".join(self.ids) + self.embedder.name + cfg.model_name + str(cfg.max_seq_len)).encode()).hexdigest()[:12]
        path = Path(cfg.cache_dir) / f"emb_{self.embedder.name}_{key}.npy"
        if path.is_file():
            m = np.load(path)
            if m.shape[0] == len(self.ids):
                return m
        texts = [embed_text(self.rows[c]) for c in self.ids]
        # 긴 것부터 묶으면 패딩이 줄어 빠름
        order = np.argsort([-len(t) for t in texts])
        vecs = self.embedder.encode([texts[i] for i in order], show_progress=True)
        m = np.zeros_like(vecs)
        m[order] = vecs
        path.parent.mkdir(parents=True, exist_ok=True)
        np.save(path, m)
        return m

    # ------------------------------------------------------------
    def _ranks(self, scores: np.ndarray, mask: np.ndarray, n: int) -> list[int]:
        idx = np.where(mask)[0]
        if not len(idx):
            return []
        return [int(i) for i in idx[np.argsort(-scores[idx], kind="stable")[:n]]]

    def _search(self, q: str, qvec: np.ndarray, mask: np.ndarray, boost_cats: set[str] | None) -> list[Candidate]:
        cfg = self.cfg
        cos = self.matrix @ qvec
        fused: dict[int, float] = {}
        d_rank: dict[int, int] = {}
        b_rank: dict[int, int] = {}
        if cfg.use_dense:
            for r, i in enumerate(self._ranks(cos, mask, cfg.n_candidates), 1):
                d_rank[i] = r
                fused[i] = fused.get(i, 0.0) + 1.0 / (cfg.rrf_k + r)
        if cfg.use_bm25:
            bm = self.bm25.scores(q)
            idx = [i for i in self._ranks(bm, mask, cfg.n_candidates) if bm[i] > 0]
            for r, i in enumerate(idx, 1):
                b_rank[i] = r
                fused[i] = fused.get(i, 0.0) + 1.0 / (cfg.rrf_k + r)
        if not cfg.use_bm25 and not cfg.use_dense:
            raise ValueError("use_dense / use_bm25 중 하나는 켜야 해요")
        for i in fused:
            mult = 1.0
            if cfg.reg_boost and self.is_reg[i]:
                mult += cfg.reg_boost
            if boost_cats and self.cats[i] & boost_cats:
                mult += cfg.soft_boost
            fused[i] *= mult
        order = sorted(fused, key=lambda i: -fused[i])[: cfg.pool]
        return [
            Candidate(
                chunk_id=self.ids[i], score=fused[i], cos=float(cos[i]), dense_rank=d_rank.get(i), bm25_rank=b_rank.get(i),
                doc_kind=self.rows[self.ids[i]].get("doc_kind") or "", categories=sorted(self.cats[i]),
                in_filter=bool(self.cats[i] & boost_cats) if boost_cats else bool(mask[i]),
            )
            for i in order
        ]

    def search(self, q: str, qc: QueryClass | None = None, qvec: np.ndarray | None = None) -> SearchResult:
        cfg = self.cfg
        if qvec is None:
            qvec = self.embedder.encode([q], is_query=True)[0]
        everything = np.ones(len(self.ids), dtype=bool)
        cats = set(qc.category_names) | {COMMON} if qc else set()
        mode = cfg.filter_mode if qc else "none"
        if mode == "adaptive":
            mode = "hard" if qc.confident else "soft"

        used_fallback = False
        if mode == "hard":
            mask = np.array([bool(c & cats) for c in self.cats])
            cands = self._search(q, qvec, mask, None)
            top_cos = max((c.cos for c in cands), default=0.0)
            if cfg.filter_mode == "adaptive" and (len(cands) < cfg.min_filtered or top_cos < cfg.fallback_cos):
                cands = self._search(q, qvec, everything, cats)       # 필터가 너무 좁음 → 해제(소프트로)
                used_fallback, mode = True, "soft"
        elif mode == "soft":
            cands = self._search(q, qvec, everything, cats)
        else:
            cands = self._search(q, qvec, everything, None)
        return SearchResult(
            candidates=cands, used_filter=mode != "none", filter_categories=sorted(cats) if mode != "none" else [],
            filter_kind=mode, used_fallback=used_fallback, top_cos=round(max((c.cos for c in cands), default=0.0), 4),
            doc_kind_counts=dict(Counter(c.doc_kind for c in cands)),
        )

In [ ]:
%%writefile rag/pipeline.py
"""질문 하나 → ① 쪼개기 → ② 분류 → ③ 라우터 → ④ 검색 → LlmRequest.

LLM 호출과 HR DB 조회는 여기서 하지 않는다. hr_needed=True 면 API 쪽에서 HrContext 를 채워 넣으면 된다.
"""

from __future__ import annotations

from dataclasses import dataclass, field

from .classify import Classifier, QueryClass
from .embed import make_embedder
from .goldenset import ChunkHit, LlmRequest, Route, chunk_hit_from_row, route
from .retrieve import Retriever, SearchResult
from .split import split_question

# 라우트 이름
SEARCH, SEARCH_HR, CLARIFY, OUT, CONTACT, TEAM = "검색", "검색+HR", "되묻기", "범위밖", "담당자연결", "권한확인"


@dataclass
class SubResult:
    question: str
    query_class: QueryClass
    route: str
    search: SearchResult | None


@dataclass
class PipelineResult:
    question: str
    status: str                        # 검색 / 검색+HR / 되묻기 / 범위밖 / 담당자연결 / 권한확인
    message: str | None
    hr_needed: bool
    team_scope: bool
    hr_route: Route
    subs: list[SubResult]
    chunks: list[ChunkHit] = field(default_factory=list)
    llm_request: LlmRequest | None = None


def _sub_route(qc: QueryClass, hr: Route) -> str:
    if not qc.in_scope:
        return OUT
    if qc.sensitive:
        return CONTACT
    if hr.team_scope:
        return TEAM
    if qc.ambiguous:
        return CLARIFY
    return SEARCH_HR if hr.hr else SEARCH


class Pipeline:
    def __init__(self, cfg, embedder=None, _retriever=None, _classifier=None):
        self.cfg = cfg
        self.embedder = embedder or make_embedder(cfg)
        self.retriever = _retriever or Retriever(cfg, self.embedder)
        self.classifier = _classifier or Classifier(self.embedder, cfg)

    def with_config(self, cfg) -> "Pipeline":
        """모델·색인은 그대로, 스위치(검색 방식·필터)만 바꾼 파이프라인."""
        return Pipeline(cfg, self.embedder, self.retriever.with_config(cfg), self.classifier.with_config(cfg))

    # ------------------------------------------------------------
    def run(self, question: str) -> PipelineResult:
        question = question.strip()
        hr = route(question)
        subs: list[SubResult] = []
        parts = split_question(question)
        vecs = self.embedder.encode(parts, is_query=True)
        for q, v in zip(parts, vecs):
            qc = self.classifier(q, v)
            rt = _sub_route(qc, hr)
            sr = self.retriever.search(q, qc, v) if rt not in (OUT,) else None
            subs.append(SubResult(q, qc, rt, sr))

        routes = [s.route for s in subs]
        message = None
        if all(r == OUT for r in routes):
            status, message = OUT, "인사·복리후생 규정에 관한 질문만 답할 수 있어요."
        elif CONTACT in routes:
            status, message = CONTACT, "민감한 사안이라 담당 부서(인사팀·고충상담 창구) 연결을 안내해요. 관련 규정은 같이 찾아 둬요."
        elif TEAM in routes:
            status, message = TEAM, "팀·부서 단위 데이터는 다른 사람 정보라 권한 확인이 필요해요 (팀장·인사팀만 조회)."
        elif len(subs) == 1 and routes[0] == CLARIFY:
            opts = subs[0].query_class.clarify_options
            status, message = CLARIFY, "어떤 것을 말씀하시는 건가요? " + " / ".join(opts)
        else:
            status = SEARCH_HR if hr.hr else SEARCH
            if hr.hr:
                message = f"HR 조회 필요 ({hr.reason})"

        res = PipelineResult(question, status, message, hr.hr, hr.team_scope, hr, subs)
        if status in (OUT, CLARIFY):
            return res
        res.chunks = self._merge([s.search for s in subs if s.search])
        res.llm_request = LlmRequest(question=question, chunks=res.chunks)
        return res

    def _merge(self, results: list[SearchResult]) -> list[ChunkHit]:
        """하위 질문들의 후보를 번갈아 가며 합친다 (각 질문의 1등부터). 최대 top_k 개."""
        picked: list = []
        seen: set[str] = set()
        depth = max((len(r.candidates) for r in results), default=0)
        for d in range(depth):
            for r in results:
                if d < len(r.candidates) and r.candidates[d].chunk_id not in seen:
                    seen.add(r.candidates[d].chunk_id)
                    picked.append(r.candidates[d])
        picked = picked[: self.cfg.top_k]
        rows = self.retriever.rows
        return [
            chunk_hit_from_row(rows[c.chunk_id], score=max(-1.0, min(1.0, c.cos)), rank=i)
            for i, c in enumerate(picked, 1)
        ]

## 2. 파이프라인 구축

In [ ]:
import importlib, sys
for m in [m for m in sys.modules if m == "rag" or m.startswith("rag.")]:
    del sys.modules[m]                       # 위에서 코드를 고쳤으면 새로 불러오기

from rag import Pipeline, load_config, split_question

cfg = load_config()          # 기본 구성 C
pipe = Pipeline(cfg)         # BGE-M3 불러오기 + 임베딩(처음만) + BM25 색인
print("임베딩 장치:", pipe.embedder.device, "| 검색 대상 조각 수:", len(pipe.retriever.rows))

## ① 질문 쪼개기

In [ ]:
for q in ["결혼하면 휴가 며칠이고 경조금 얼마야?",
          "5일 썼는데, 며칠까지 돼요?",
          "복지포인트는 언제 들어오고 어디서 쓸 수 있어요?"]:
    print(q, "→", split_question(q))

## ②③ 분류기와 라우터

In [ ]:
q = "결혼하면 휴가 며칠이고 경조금 얼마야?"
r = pipe.run(q)
print("상태:", r.status, r.message or "")
for s in r.subs:
    print(s.question)
    print("  카테고리:", [(c, round(x, 3)) for c, x in s.query_class.categories], "confident =", s.query_class.confident)
    print("  전체 점수:", {c: round(x, 3) for c, x in sorted(s.query_class.scores.items(), key=lambda kv: -kv[1])})
    print("  slots:", s.query_class.slots, "| 라우터:", s.route)

## ④ 검색 후보 (최대 20개)

In [ ]:
import pandas as pd

s = r.subs[0]
print("필터:", s.search.used_filter, s.search.filter_categories, "| fallback:", s.search.used_fallback,
      "| 1등 코사인:", s.search.top_cos, "|", s.search.doc_kind_counts)
df = pd.DataFrame([c.to_dict() for c in s.search.candidates])
df.insert(1, "title", [pipe.retriever.rows[c]["doc_name"] + " " + (pipe.retriever.rows[c]["article"] or "") for c in df.chunk_id])
df

## LLM 에 넘길 것: ChunkHit 8개 → LlmRequest

In [ ]:
from rag.goldenset import build_messages

for c in r.chunks:
    print(c.rank, c.chunk_id, c.title, round(c.score, 3), c.categories)
print()
print(build_messages(r.llm_request)[1]["content"][:800])

## 실험 구성 바꿔 보기 (A0 / A1 / B / C)

모델과 색인은 그대로 두고 스위치만 바꿔요.

| 구성 | 검색 |
|---|---|
| A0 | Dense만 |
| A1 | + BM25, RRF로 합치기 |
| B | + 분류 결과로 하드 필터 |
| C | + confidence 따라 하드/소프트 필터, 약하면 필터 해제 (기본) |

In [ ]:
q = "서울 출장 가면 숙박비 얼마까지 나와?"
for name in ["A0", "A1", "B", "C"]:
    rr = pipe.with_config(cfg.preset(name)).run(q)
    print(name, [c.chunk_id for c in rr.chunks[:5]])

## 범위 밖 · 애매함 · 팀 단위

In [ ]:
for q in ["요즘 주식 뭐 사야 돼?", "휴가 며칠이야?", "우리 팀 연차 현황 보여줘", "나 올해 연차 몇 개야?"]:
    rr = pipe.run(q)
    print(f"{q} → {rr.status} | hr_needed={rr.hr_needed} | {rr.message or ''}")